In [ ]:
import os
import sys
from pathlib import Path
import nbformat
from flask import Flask, jsonify, request
from flask_cors import CORS

# Get the project root directory
# When running as notebook, Path.cwd() should be the backend directory
# So project root is parent of cwd
current_dir = Path.cwd()
if current_dir.name == "backend":
    project_root = current_dir.parent
else:
    # Fallback: assume we're in project root
    project_root = current_dir

backend_dir = project_root / "backend"
init_notebook_path = backend_dir / "lib" / "data" / "init.ipynb"

# Load and execute the init notebook
nb = nbformat.read(str(init_notebook_path), as_version=4)
namespace = {}
for cell in nb.cells:
    if cell.cell_type == "code":
        # Replace relative path with absolute path
        code = cell.source
        if "../../../data/data.h5" in code:
            data_path = project_root / "data" / "data.h5"
            # Replace the path string, handling both quoted and unquoted cases
            import re
            # Replace "path" or 'path' with the absolute path
            code = re.sub(r'["\']\.\.\/\.\.\/\.\.\/data\/data\.h5["\']', f'"{str(data_path)}"', code)
        exec(code, namespace)

# Extract functions and variables we need
create_plant = namespace.get('create_plant')
create_empID = namespace.get('create_empID')
hruuid = namespace.get('hruuid')
h5py = namespace.get('h5py')
datetime = namespace.get('datetime')

# Initialize Flask app
# Provide explicit root_path and import_name since we're executing from a notebook
import os
app = Flask('server', root_path=str(backend_dir))
CORS(app, origins=["http://localhost:4321", "http://localhost:3000"])

# HDF5 file path
data_file_path = project_root / "data" / "data.h5"


In [ ]:
def get_h5_file():
    """Open and return the HDF5 file handle"""
    return h5py.File(str(data_file_path), "a")

def create_plant_with_context(f):
    """Create a plant using the provided file handle"""
    # Set up the context that create_plant expects
    namespace['file'] = f
    namespace['plant_group'] = f.require_group("plants")
    # Execute create_plant in the updated namespace
    exec('plant = create_plant()', namespace)
    return namespace['plant']

def create_sequencer_effect(f, effect_type: str, row: int, col: int, properties: dict = None):
    """Create a sequencer effect group in sequencer_effects_properties and return its UUID"""
    effects_props_group = f.require_group("sequencer_effects_properties")
    effect_uuid = hruuid.generate()
    sequencer_effect = effects_props_group.require_group(effect_uuid)
    sequencer_effect.attrs["effect_type"] = effect_type
    sequencer_effect.attrs["sequencer_row"] = row
    sequencer_effect.attrs["sequencer_col"] = col
    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
    
    # Store properties as attributes
    if properties:
        for key, value in properties.items():
            sequencer_effect.attrs[f"prop_{key}"] = value
    
    # Return the UUID
    return effect_uuid

def get_sequencer_grid():
    """Read sequencer dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            # Create sequencer if it doesn't exist
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_sequencer_grid(grid):
    """Write 2x12 grid to sequencer dataset"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]

def get_effects_grid():
    """Read sequencer_effects dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            # Create sequencer_effects if it doesn't exist
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_effects_grid(grid):
    """Write 2x12 grid to sequencer_effects dataset"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]


In [ ]:
@app.route('/api/plants', methods=['POST'])
def create_plant_endpoint():
    """Create a new plant"""
    try:
        f = get_h5_file()
        plant = create_plant_with_context(f)
        plant_id = plant.name.split('/')[-1]  # Get the plant ID from the group name
        timestamp = plant.attrs.get("added_timestamp", "")
        f.close()
        
        return jsonify({
            "id": plant_id,
            "added_timestamp": timestamp
        }), 201
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

@app.route('/api/plants', methods=['GET'])
def list_plants():
    """List all plants"""
    try:
        plants = []
        with get_h5_file() as f:
            if "plants" in f.keys():
                plant_group = f["plants"]
                for plant_id in plant_group.keys():
                    plant = plant_group[plant_id]
                    plants.append({
                        "id": plant_id,
                        "added_timestamp": plant.attrs.get("added_timestamp", "")
                    })
        return jsonify({"plants": plants}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['GET'])
def get_sequencer():
    """Get current sequencer state"""
    try:
        grid = get_sequencer_grid()
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['PUT'])
def update_sequencer():
    """Update sequencer position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        plant_id = data.get('plant_id', '')  # Empty string to clear
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        grid = get_sequencer_grid()
        grid[row][col] = plant_id
        set_sequencer_grid(grid)
        
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['GET'])
def get_effects():
    """Get current effects grid state - returns effect types by looking up UUIDs from sequencer_effects_properties"""
    try:
        grid = get_effects_grid()
        # Convert UUIDs to effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for row in grid:
                effect_types_row = []
                for uuid in row:
                    if uuid:
                        # Look up effect type from sequencer_effects_properties group
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            # UUID not found, clear it
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['PUT'])
def update_effects():
    """Update effects grid position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        effect = data.get('effect', '')  # Empty string to clear
        properties = data.get('properties', {})  # Optional properties
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        # Validate effect type
        valid_effects = ['AC', 'DC', 'AMF', 'CMF', '']
        if effect not in valid_effects:
            return jsonify({"error": f"effect must be one of {valid_effects}"}), 400
        
        grid = get_effects_grid()
        
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            
            # If clearing effect, delete the sequencer effect group and clear the grid position
            if not effect:
                old_uuid = grid[row][col]
                if old_uuid and old_uuid in effects_props_group.keys():
                    del effects_props_group[old_uuid]
                grid[row][col] = ""
            else:
                # Check if there's already an effect at this position
                old_uuid = grid[row][col]
                
                # If there's an existing effect, update it instead of creating new
                if old_uuid and old_uuid in effects_props_group.keys():
                    sequencer_effect = effects_props_group[old_uuid]
                    # Update effect type and properties
                    sequencer_effect.attrs["effect_type"] = effect
                    sequencer_effect.attrs["sequencer_row"] = row
                    sequencer_effect.attrs["sequencer_col"] = col
                    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
                    
                    # Update properties
                    if properties:
                        for key, value in properties.items():
                            sequencer_effect.attrs[f"prop_{key}"] = value
                    
                    # Keep the same UUID
                    grid[row][col] = old_uuid
                else:
                    # Create new sequencer effect group
                    uuid = create_sequencer_effect(f, effect, row, col, properties)
                    grid[row][col] = uuid
        
        set_effects_grid(grid)
        
        # Return effect types for frontend compatibility
        effect_types_grid = []
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            for grid_row in grid:
                effect_types_row = []
                for uuid in grid_row:
                    if uuid:
                        if uuid in effects_props_group.keys():
                            sequencer_effect = effects_props_group[uuid]
                            effect_type = sequencer_effect.attrs.get("effect_type", "")
                            effect_types_row.append(effect_type)
                        else:
                            effect_types_row.append("")
                    else:
                        effect_types_row.append("")
                effect_types_grid.append(effect_types_row)
        
        return jsonify({"effects": effect_types_grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['GET'])
def get_effect_properties(row, col):
    """Get properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"properties": {}}), 200
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"properties": {}}), 200
            
            sequencer_effect = effects_props_group[uuid]
            properties = {}
            
            # Extract all prop_* attributes
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]  # Remove "prop_" prefix
                    properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['PUT'])
def update_effect_properties(row, col):
    """Update properties for an effect at a specific position"""
    try:
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        data = request.get_json()
        properties = data.get('properties', {})
        
        with get_h5_file() as f:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"error": "Effect not found at this position"}), 404
            
            effects_props_group = f.require_group("sequencer_effects_properties")
            if uuid not in effects_props_group.keys():
                return jsonify({"error": "Effect not found at this position"}), 404
            
            sequencer_effect = effects_props_group[uuid]
            
            # Update properties
            for key, value in properties.items():
                sequencer_effect.attrs[f"prop_{key}"] = value
            
            # Return updated properties
            updated_properties = {}
            for key in sequencer_effect.attrs.keys():
                if key.startswith("prop_"):
                    prop_name = key[5:]
                    updated_properties[prop_name] = sequencer_effect.attrs[key]
            
            return jsonify({
                "effect_type": sequencer_effect.attrs.get("effect_type", ""),
                "properties": updated_properties
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
import threading
import time
import math

# Tile allocation constants based on paper
# First 8 columns (0-7): Storage/Observation area  
# Last 4 columns (8-11): EM exposure zones (swap space available)
STORAGE_COLS = list(range(0, 8))  # Columns 0-7 for storage
EM_EXPOSURE_COLS = list(range(8, 12))  # Columns 8-11 for EM exposure

# Robot configuration: 2 robots on each side (4 total)
# Left robots (0,1) serve row 0, Right robots (2,3) serve row 1
ROBOT_SPEED = 0.5  # columns per tick (simulated speed)
OBSERVATION_TICKS = 3  # ticks to observe a plant
PICKUP_TICKS = 2  # ticks to pick up a plant
PUTDOWN_TICKS = 2  # ticks to put down a plant
RECENT_OBSERVATION_LIMIT = 20  # number of recent observations to track for scheduling

# Simulation state (in-memory for now)
simulation_state = {
    'running': False,
    'tick': 0,
    'speed': 1.0,  # ticks per second
    'robots': [
        {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 1, 'row': 0, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 2, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 3, 'row': 1, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
    ],
    'observation_queue': [],  # Plants queued for observation
    'observation_station': {'row': -1, 'col': 6},  # Camera position (outside grid)
    'plants_observed': [],  # Plants that have been observed this cycle
    'last_tick_time': None
}

simulation_lock = threading.Lock()
simulation_thread = None

def get_tile_allocation():
    """Return tile allocation info for frontend"""
    return {
        'storage_cols': STORAGE_COLS,
        'em_exposure_cols': EM_EXPOSURE_COLS,
        'observation_station': simulation_state['observation_station']
    }

def calculate_travel_time(from_col, to_col):
    """Calculate ticks needed to travel between columns"""
    distance = abs(to_col - from_col)
    return math.ceil(distance / ROBOT_SPEED)

def find_available_robot(row):
    """Find an idle robot for the given row"""
    for robot in simulation_state['robots']:
        if robot['row'] == row and robot['state'] == 'idle':
            return robot
    return None

def update_robot(robot):
    """Update a single robot's state for one tick"""
    if robot['state'] == 'idle':
        return
    
    if robot['state'] == 'moving_to_pickup':
        # Move toward target
        if robot['target_col'] is not None:
            if robot['col'] < robot['target_col']:
                robot['col'] = min(robot['col'] + ROBOT_SPEED, robot['target_col'])
            elif robot['col'] > robot['target_col']:
                robot['col'] = max(robot['col'] - ROBOT_SPEED, robot['target_col'])
            
            if robot['col'] == robot['target_col']:
                robot['state'] = 'picking_up'
                robot['ticks_remaining'] = PICKUP_TICKS
    
    elif robot['state'] == 'picking_up':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Pick up complete - get plant from grid
            grid = get_sequencer_grid()
            col = round(robot['target_col'])
            if 0 <= col < 12 and grid[robot['row']][col]:
                robot['holding_plant'] = grid[robot['row']][col]
                grid[robot['row']][col] = ''
                set_sequencer_grid(grid)
            robot['state'] = 'moving_to_observe'
            robot['target_col'] = simulation_state['observation_station']['col']
    
    elif robot['state'] == 'moving_to_observe':
        target = simulation_state['observation_station']['col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'observing'
            robot['ticks_remaining'] = OBSERVATION_TICKS
    
    elif robot['state'] == 'observing':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Observation complete
            if robot['holding_plant']:
                simulation_state['plants_observed'].append({
                    'plant_id': robot['holding_plant'],
                    'tick': simulation_state['tick'],
                    'robot_id': robot['id']
                })
            robot['state'] = 'moving_to_return'
            # Return to original position
            robot['target_col'] = robot.get('original_col', 0)
    
    elif robot['state'] == 'moving_to_return':
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'putting_down'
            robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'putting_down':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Put down complete - return plant to grid
            if robot['holding_plant']:
                grid = get_sequencer_grid()
                col = round(robot['target_col'])
                if 0 <= col < 12 and not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    set_sequencer_grid(grid)
                robot['holding_plant'] = None
            robot['state'] = 'idle'
            robot['target_col'] = None

def assign_observation_task():
    """Try to assign observation tasks to idle robots"""
    grid = get_sequencer_grid()
    
    for row in range(2):
        robot = find_available_robot(row)
        if not robot:
            continue
        
        # Find a plant in storage area that hasn't been observed recently
        observed_ids = [p['plant_id'] for p in simulation_state['plants_observed'][-RECENT_OBSERVATION_LIMIT:]]
        
        for col in STORAGE_COLS:
            plant_id = grid[row][col]
            if plant_id and plant_id not in observed_ids:
                # Assign this plant for observation
                robot['state'] = 'moving_to_pickup'
                robot['target_col'] = float(col)
                robot['original_col'] = float(col)  # Remember where to return
                break

def simulation_tick():
    """Execute one simulation tick"""
    with simulation_lock:
        if not simulation_state['running']:
            return
        
        simulation_state['tick'] += 1
        simulation_state['last_tick_time'] = time.time()
        
        # Update all robots
        for robot in simulation_state['robots']:
            update_robot(robot)
        
        # Try to assign new tasks
        assign_observation_task()

def simulation_loop():
    """Main simulation loop running in background thread"""
    while True:
        with simulation_lock:
            if not simulation_state['running']:
                break
            speed = simulation_state['speed']
        
        simulation_tick()
        time.sleep(1.0 / speed if speed > 0 else 1.0)

def start_simulation():
    """Start the simulation"""
    global simulation_thread
    with simulation_lock:
        if simulation_state['running']:
            return False
        simulation_state['running'] = True
    
    simulation_thread = threading.Thread(target=simulation_loop, daemon=True)
    simulation_thread.start()
    return True

def stop_simulation():
    """Stop the simulation"""
    with simulation_lock:
        simulation_state['running'] = False
    return True

def reset_simulation():
    """Reset simulation state"""
    with simulation_lock:
        simulation_state['running'] = False
        simulation_state['tick'] = 0
        simulation_state['speed'] = 1.0
        simulation_state['robots'] = [
            {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 1, 'row': 0, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 2, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 3, 'row': 1, 'col': 3.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
        ]
        simulation_state['observation_queue'] = []
        simulation_state['plants_observed'] = []
        simulation_state['last_tick_time'] = None
    return True


In [ ]:
# Simulation API endpoints

@app.route('/api/simulation/state', methods=['GET'])
def get_simulation_state():
    """Get current simulation state"""
    try:
        with simulation_lock:
            return jsonify({
                'running': simulation_state['running'],
                'tick': simulation_state['tick'],
                'speed': simulation_state['speed'],
                'robots': simulation_state['robots'],
                'plants_observed': simulation_state['plants_observed'][-10:],  # Last 10
                'tile_allocation': get_tile_allocation()
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/start', methods=['POST'])
def start_simulation_endpoint():
    """Start the simulation"""
    try:
        started = start_simulation()
        return jsonify({'success': started, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/pause', methods=['POST'])
def pause_simulation_endpoint():
    """Pause the simulation"""
    try:
        stopped = stop_simulation()
        return jsonify({'success': stopped, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/reset', methods=['POST'])
def reset_simulation_endpoint():
    """Reset the simulation"""
    try:
        reset = reset_simulation()
        return jsonify({'success': reset, 'tick': 0}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/speed', methods=['PUT'])
def set_simulation_speed():
    """Set simulation speed (ticks per second)"""
    try:
        data = request.get_json()
        speed = data.get('speed', 1.0)
        
        # Clamp speed between 0.1 and 10.0
        speed = max(0.1, min(10.0, float(speed)))
        
        with simulation_lock:
            simulation_state['speed'] = speed
        
        return jsonify({'speed': speed}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/tick', methods=['POST'])
def manual_tick():
    """Manually advance one tick (for debugging/testing)"""
    try:
        with simulation_lock:
            was_running = simulation_state['running']
            simulation_state['running'] = True
        
        simulation_tick()
        
        with simulation_lock:
            simulation_state['running'] = was_running
            return jsonify({
                'tick': simulation_state['tick'],
                'robots': simulation_state['robots']
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/tile-allocation', methods=['GET'])
def get_tile_allocation_endpoint():
    """Get tile allocation info"""
    try:
        return jsonify(get_tile_allocation()), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
if __name__ == "__main__":
    print(f"Starting Flask server on http://localhost:5000")
    print(f"API endpoints:")
    print(f"  POST /api/plants - Create a new plant")
    print(f"  GET /api/plants - List all plants")
    print(f"  GET /api/sequencer - Get sequencer grid")
    print(f"  PUT /api/sequencer - Update sequencer position")
    print(f"  GET /api/effects - Get effects grid")
    print(f"  PUT /api/effects - Update effects grid position")
    print(f"  GET /api/effects/<row>/<col>/properties - Get effect properties")
    print(f"  PUT /api/effects/<row>/<col>/properties - Update effect properties")
    print(f"  GET /api/simulation/state - Get simulation state")
    print(f"  POST /api/simulation/start - Start simulation")
    print(f"  POST /api/simulation/pause - Pause simulation")
    print(f"  POST /api/simulation/reset - Reset simulation")
    print(f"  PUT /api/simulation/speed - Set simulation speed")
    print(f"  POST /api/simulation/tick - Manual tick")
    print(f"  GET /api/tile-allocation - Get tile allocation")
    app.run(host='0.0.0.0', port=5000, debug=True)
